In [1]:
import re
import pandas as pd

In [2]:
df = pd.read_csv("../datasets/final_dataset.csv")

In [3]:
df = df.copy()

In [4]:
def clean_text(text):

    # Convert text to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove email addresses
    text = re.sub(r"\S+@\S+", "", text)

    # Remove special characters and punctuation
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [5]:
df["cleaned_text"] = df["complaint_text"].apply(clean_text)

In [6]:
df[["complaint_text", "cleaned_text"]].head(10)

,complaint_text,cleaned_text
0,Product setup. I'm having an issue with the {p...,product setup i m having an issue with the pro...
1,Peripheral compatibility. I'm having an issue ...,peripheral compatibility i m having an issue w...
2,Network problem. I'm facing a problem with my ...,network problem i m facing a problem with my p...
3,Account access. I'm having an issue with the {...,account access i m having an issue with the pr...
4,Data loss. I'm having an issue with the {produ...,data loss i m having an issue with the product...
5,Payment issue. I'm facing a problem with my {p...,payment issue i m facing a problem with my pro...
6,Refund request. I'm unable to access my {produ...,refund request i m unable to access my product...
7,Battery life. I'm having an issue with the {pr...,battery life i m having an issue with the prod...
8,Installation support. I'm having an issue with...,installation support i m having an issue with ...
9,Payment issue. My {product_purchased} is makin...,payment issue my product purchased is making s...


In [7]:
# %% Tokenization

df["tokens"] = df["cleaned_text"].apply(
    lambda text: text.split()
)

print(df[["cleaned_text", "tokens"]].head())

                                        cleaned_text  \
0  product setup i m having an issue with the pro...   
1  peripheral compatibility i m having an issue w...   
2  network problem i m facing a problem with my p...   
3  account access i m having an issue with the pr...   
4  data loss i m having an issue with the product...   

                                              tokens  
0  [product, setup, i, m, having, an, issue, with...  
1  [peripheral, compatibility, i, m, having, an, ...  
2  [network, problem, i, m, facing, a, problem, w...  
3  [account, access, i, m, having, an, issue, wit...  
4  [data, loss, i, m, having, an, issue, with, th...  


In [8]:
# %% Stopword removal

import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

# Preserve important negation words
negation_words = {"no", "not", "nor", "never"}

stop_words = stop_words - negation_words

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
df["tokens_without_stopwords"] = df["tokens"].apply(
    lambda words: [
        word for word in words
        if word not in stop_words
    ]
)

print(df[["tokens", "tokens_without_stopwords"]].head())



                                              tokens  \
0  [product, setup, i, m, having, an, issue, with...   
1  [peripheral, compatibility, i, m, having, an, ...   
2  [network, problem, i, m, facing, a, problem, w...   
3  [account, access, i, m, having, an, issue, wit...   
4  [data, loss, i, m, having, an, issue, with, th...   

                            tokens_without_stopwords  
0  [product, setup, issue, product, purchased, pl...  
1  [peripheral, compatibility, issue, product, pu...  
2  [network, problem, facing, problem, product, p...  
3  [account, access, issue, product, purchased, p...  
4  [data, loss, issue, product, purchased, please...  


In [10]:
df["final_text"] = df["tokens_without_stopwords"].apply(lambda words:" ".join(words) )
print(df[["complaint_text","final_text"]])

                                         complaint_text  \
0     Product setup. I'm having an issue with the {p...   
1     Peripheral compatibility. I'm having an issue ...   
2     Network problem. I'm facing a problem with my ...   
3     Account access. I'm having an issue with the {...   
4     Data loss. I'm having an issue with the {produ...   
...                                                 ...   
8464  Installation support. My {product_purchased} i...   
8465  Refund request. I'm having an issue with the {...   
8466  Account access. I'm having an issue with the {...   
8467  Payment issue. I'm having an issue with the {p...   
8468  Hardware issue. There seems to be a hardware p...   

                                             final_text  
0     product setup issue product purchased please a...  
1     peripheral compatibility issue product purchas...  
2     network problem facing problem product purchas...  
3     account access issue product purchased please ...  
4

In [11]:
# %% Check empty processed text

empty_count = df["final_text"].str.strip().eq("").sum()

print("Empty processed complaints:", empty_count)

Empty processed complaints: 0


In [12]:
# %% Save processed dataset

df.to_csv(
    "../datasets/complaints_preprocessed.csv",
    index=False
)

print("Preprocessed dataset saved successfully!")

Preprocessed dataset saved successfully!


In [13]:
print(df.head())

   complaint_id                                     complaint_text  \
0             1  Product setup. I'm having an issue with the {p...   
1             2  Peripheral compatibility. I'm having an issue ...   
2             3  Network problem. I'm facing a problem with my ...   
3             4  Account access. I'm having an issue with the {...   
4             5  Data loss. I'm having an issue with the {produ...   

          category  priority sentiment    emotion         department  \
0  Technical issue  Critical  Positive  Satisfied  Technical Support   
1  Technical issue  Critical  Positive      Other  Technical Support   
2  Technical issue       Low  Positive      Other  Technical Support   
3  Billing inquiry       Low  Positive  Satisfied            Billing   
4  Billing inquiry       Low  Negative      Other            Billing   

                                        cleaned_text  \
0  product setup i m having an issue with the pro...   
1  peripheral compatibility i m ha